# UGM Academic Policy Intelligence — Kaggle GPU Training & Evaluation Pipeline

Notebook ini menjalankan **seluruh komputasi berat** di GPU Kaggle (T4 / P100 16GB VRAM):
1. Ekstrak/salin bundle dataset `ugm-policy-kaggle-bundle` dari `/kaggle/input/...`
2. Instal dependensi GPU tanpa menimpa CUDA `torch` bawaan Kaggle
3. Fine-tuning **B5 QLoRA 4-bit** (`Qwen/Qwen2.5-1.5B-Instruct`) hanya pada `data/benchmark/train.jsonl`
4. Menjalankan **Ablasi A1–A7**, **Perbandingan B0–B6 (3 run independen)** dengan neural dense (`intfloat/multilingual-e5-small`) + cross-encoder reranker (`BAAI/bge-reranker-base`) + open-weight LLM, serta **Red-Team Suite (20 kasus)**
5. Menyimpan hasil dan adapter ke `/kaggle/working/kaggle_outputs.zip` & `/kaggle/working/reports_kaggle/`.

In [ ]:
import os, zipfile, glob, shutil

WORK_DIR = '/kaggle/working/ugmPolicy'
os.makedirs(WORK_DIR, exist_ok=True)

zips = glob.glob('/kaggle/input/**/ugm_policy_kaggle_bundle.zip', recursive=True)
if zips:
    print('Extracting zip:', zips[0])
    with zipfile.ZipFile(zips[0], 'r') as zf:
        zf.extractall(WORK_DIR)
else:
    corpus_hits = glob.glob('/kaggle/input/**/corpus_clean.jsonl', recursive=True)
    assert corpus_hits, 'Dataset ugm-policy-kaggle-bundle tidak ditemukan di /kaggle/input!'
    src_root = os.path.dirname(corpus_hits[0])
    print('Copying dataset dir:', src_root)
    shutil.copytree(src_root, WORK_DIR, dirs_exist_ok=True)

%cd /kaggle/working/ugmPolicy
%env CUDA_VISIBLE_DEVICES=0
!nvidia-smi

In [ ]:
# Instal/upgrade library HF/PEFT/QLoRA tanpa menimpa CUDA torch bawaan Kaggle
!pip install -q -U transformers peft bitsandbytes trl accelerate datasets sentence-transformers rank-bm25 pymupdf pydantic pyyaml

In [ ]:
# 1. Siapkan SFT train.jsonl (strictly leak-free) & latih B5 QLoRA (4-bit NF4) di GPU Kaggle
import subprocess, sys
def run_stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

run_stream([
    sys.executable, '-u', 'scripts/train_qlora_kaggle.py',
    '--base-model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--output-dir', 'models/b5_qlora_adapter',
    '--epochs', '3',
])

In [ ]:
# 2. Jalankan Ablasi A1-A7 (dev), B0-B6 x 3 seeds (locked test), & Red-Team Suite dengan neural dense + reranker + Qwen2.5-1.5B
run_stream([
    sys.executable, '-u', 'scripts/run_experiments.py',
    '--backend', 'hf',
    '--base-model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--adapter-dir', 'models/b5_qlora_adapter',
    '--dense-model', 'intfloat/multilingual-e5-small',
    '--reranker-model', 'BAAI/bge-reranker-base',
    '--seeds', '101', '202', '303',
    '--out-dir', 'reports_kaggle',
])

In [ ]:
# 3. Inspect ringkasan & salin output ke /kaggle/working agar mudah ditarik via kaggle kernels output
import json, shutil
print('=== B0-B6 Summary ===')
print(open('reports_kaggle/b0_b6_summary.json').read()[:3000])
print('=== Red-Team Summary ===')
rt = json.load(open('reports_kaggle/red_team_report.json'))
print({'total': rt['total_fixtures'], 'baseline_pass': rt['baseline_pass_rate'], 'mitigated_pass': rt['mitigated_pass_rate']})

shutil.copytree('reports_kaggle', '/kaggle/working/reports_kaggle', dirs_exist_ok=True)
shutil.copytree('models/b5_qlora_adapter', '/kaggle/working/b5_qlora_adapter', dirs_exist_ok=True)
os.makedirs('/kaggle/working/export_bundle', exist_ok=True)
shutil.copytree('reports_kaggle', '/kaggle/working/export_bundle/reports_kaggle', dirs_exist_ok=True)
shutil.copytree('models/b5_qlora_adapter', '/kaggle/working/export_bundle/models/b5_qlora_adapter', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/kaggle_outputs', 'zip', '/kaggle/working/export_bundle')
shutil.rmtree('/kaggle/working/export_bundle', ignore_errors=True)
print('Siap diunduh: /kaggle/working/kaggle_outputs.zip & /kaggle/working/reports_kaggle/')